<a href="https://colab.research.google.com/github/opatil31/sc_mechinterp-op/blob/main/MI_SparseSpike_and_SlabVAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Spike-and-Slab VAE for Mechanistic Interpretability

Author - Oankar Patil:
My motivation for implementing this is because I think one of the reasons SAEs are being outperformed by linear probes
is that we're trying to learn sparse dictionaries of what's really a continuous feature space

So to try to fix that I thought I'd implement a Sparse VAE (overcomplete). From a bayesian persp, a laplace prior was an initial direction
I considered, but I don't think that makes sense because it's fundamentally a shrinkage prior.

Ideally, we'd want clean on/off and not lots of non-zero small activations, which is what a laplace direction would likely give us.

So effectively, we want something that seperates selectioni itself from magnitude:
- We could use a hard-Concrete
- We could.. learn sparsity via beta-bernouli hyperprior (so this way the model decides how many features to actually use)

So with this kind of intuition in mind, I think we'd manage to get actual selection sparsity but still keep continuous values where they're active

So specifically this implementation features:
- Fixed Bernoulli spike prior on binary gates s_i (sparsity is a hyperparameter)
- Gaussian slab prior on continuous latents z-tilde_i
- Hard-concrete relaxation for diff sampling
- Overcomplete
- Decoder orthogonality constraints and non-negative constrains (toggalble (<- is that a word lol?))
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Bernoulli, Normal, Beta, RelaxedBernoulli
from typing import Optional, Tuple, Dict
import math

class SpikeAndSlabEncoder(nn.Module):
    """
    Encoder that outputs parameters for spike-and-slab posterior:
    - Gate probabilities (spikes): q(s_i | x)
    - Gaussian parameters (slabs): q(z-tilde_i | x, s_i=1)
    """
    def __init__(
        self,
        input_dim: int,
        hidden_dims: list[int],
        latent_dim: int,
        use_hard_concrete: bool = True,
        temperature: float = 0.5,
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.use_hard_concrete = use_hard_concrete
        self.temperature = temperature

        # Build encoder network
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
            ])
            prev_dim = hidden_dim

        self.encoder = nn.Sequential(*layers)

        # Output heads
        # Gate logits (for Bernoulli spikes)
        self.gate_logits = nn.Linear(prev_dim, latent_dim)

        # Gaussian slab parameters
        self.mean = nn.Linear(prev_dim, latent_dim)
        self.logvar = nn.Linear(prev_dim, latent_dim)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Returns:
            gate_logits: Logits for Bernoulli gate distribution
            z_mean: Mean of Gaussian slab
            z_logvar: Log-variance of Gaussian slab
        """
        h = self.encoder(x)

        return {
            'gate_logits': self.gate_logits(h),
            'z_mean': self.mean(h),
            'z_logvar': self.logvar(h),
        }

    def sample(
        self,
        gate_logits: torch.Tensor,
        z_mean: torch.Tensor,
        z_logvar: torch.Tensor,
        hard: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Sample from spike-and-slab posterior using reparameterization.

        Args:
            gate_logits: Logits for gate distribution [batch, latent_dim]
            z_mean: Mean of slab distribution [batch, latent_dim]
            z_logvar: Log-variance of slab distribution [batch, latent_dim]
            hard: Whether to use hard (discrete) gates during forward pass

        Returns:
            s: Binary gates [batch, latent_dim]
            z_tilde: Continuous latents [batch, latent_dim]
            z: Combined latents z_i = s_i * z-tilde_i [batch, latent_dim]
        """
        batch_size = gate_logits.shape[0]

        if self.use_hard_concrete and self.training:
            # Hard-concrete / Gumbel-softmax relaxation for gates
            dist = RelaxedBernoulli(temperature=self.temperature, logits=gate_logits)
            s_soft = dist.rsample()

            if hard:
                # Straight-through estimator
                s_hard = (s_soft > 0.5).float()
                s = s_hard - s_soft.detach() + s_soft
            else:
                s = s_soft
        else:
            # Standard Bernoulli sampling (or hard sampling during eval)
            probs = torch.sigmoid(gate_logits)
            if self.training and not hard:
                # Soft sampling during training
                dist = RelaxedBernoulli(temperature=self.temperature, probs=probs)
                s = dist.rsample()
            else:
                # Hard sampling
                s = Bernoulli(probs=probs).sample()

        # Sample from Gaussian slab using reparameterization trick
        std = torch.exp(0.5 * z_logvar)
        eps = torch.randn_like(std)
        z_tilde = z_mean + eps * std

        # Combine: z_i = s_i * z̃_i
        z = s * z_tilde

        return s, z_tilde, z

class SpikeAndSlabDecoder(nn.Module):
    """
    Decoder for MI: a single linear projection layer.
    The columns of W (weight matrix) are the interpretable features (one per latent).
    x_recon = z @ W.T
    """
    def __init__(
        self,
        latent_dim: int,
        output_dim: int,
        orthogonality_weight: float = 0.0,
        nonnegativity: bool = False,
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.output_dim = output_dim
        self.orthogonality_weight = orthogonality_weight
        self.nonnegativity = nonnegativity

        # Linear decoder: weight shape [output_dim, latent_dim]
        self.decoder_layer = nn.Linear(latent_dim, output_dim, bias=False)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        Decode latents z [batch, latent_dim] to x_recon [batch, output_dim].
        NOTE: no forward-time ReLU on weights; nonnegativity is enforced post-step.
        """
        W = self.decoder_layer.weight  # [output_dim, latent_dim]
        x_recon = F.linear(z, W, self.decoder_layer.bias)  # z @ W.T
        return x_recon

    def apply_constraints(self):
        """
        Enforce constraints directly on decoder weights.
        Call this AFTER optimizer.step().
        - If nonnegativity: clamp weights to >= 0
        - Unit-norm **columns** (each latent feature column has ||·||_2 = 1)
        """
        with torch.no_grad():
            W = self.decoder_layer.weight  # [output_dim, latent_dim
            W_new = W

            if self.nonnegativity:
                W_new = W_new.clamp_min(0.0)

            # Unit-norm columns (dimension 0 because columns are along output_dim)
            col_norms = torch.norm(W_new, p=2, dim=0, keepdim=True)  + 1e-8# [1, latent_dim]
            W_new = W_new / col_norms
            W.copy_(W_new)

    def orthogonality_loss(self) -> torch.Tensor:
        """
        Column orthogonality: || W^T W - I ||_F^2  (features are columns of W)
        """
        W = self.decoder_layer.weight  # [output_dim, latent_dim]
        gram = W.t() @ W              # [latent_dim, latent_dim]
        identity = torch.eye(self.latent_dim, device=W.device, dtype=W.dtype)
        return torch.sum((gram - identity) ** 2)

class SpikeAndSlabVAE(nn.Module):
    def __init__(
        self,
        input_dim: int,
        latent_dim: int,
        encoder_hidden_dims: list[int] = [512, 256],
        prior_sparsity: float = 0.01,  # Fixed prior prob p(s=1)
        nonnegativity: bool = False,
        use_hard_concrete: bool = True,
        temperature: float = 0.5,
        reconstruction_loss: str = 'mse',  # 'mse' or 'bce'
    ):
        super().__init__()

        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.prior_sparsity = prior_sparsity

        self.reconstruction_loss_type = reconstruction_loss

        # Encoder
        self.encoder = SpikeAndSlabEncoder(
            input_dim=input_dim,
            hidden_dims=encoder_hidden_dims,
            latent_dim=latent_dim,
            use_hard_concrete=use_hard_concrete,
            temperature=temperature,
        )

        # Decoder
        self.decoder = SpikeAndSlabDecoder(
            latent_dim=latent_dim,
            output_dim=input_dim, # Reconstructs the input
            nonnegativity=nonnegativity,
        )

    def forward(
        self,
        x: torch.Tensor,
        hard_sampling: bool = False,
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass through VAE.

        Returns dictionary with:
            - x_recon: Reconstructed input
            - z: Combined latents
            - s: Binary gates
            - z_tilde: Continuous latents
            - gate_logits: Posterior gate logits
            - z_mean: Posterior mean
            - z_logvar: Posterior log-variance
        """
        # Encode
        encoder_output = self.encoder(x)

        # Sample latents
        s, z_tilde, z = self.encoder.sample(
            encoder_output['gate_logits'],
            encoder_output['z_mean'],
            encoder_output['z_logvar'],
            hard=hard_sampling,
        )

        # Decode
        x_recon = self.decoder(z)

        return {
            'x_recon': x_recon,
            'z': z,
            's': s,
            'z_tilde': z_tilde,
            **encoder_output,
        }

    def compute_loss(
        self,
        x: torch.Tensor,
        forward_output: Dict[str, torch.Tensor],
        beta: float = 1.0,  # This is the KL weight (like beta-VAE)
    ) -> Dict[str, torch.Tensor]:
        """
        Compute ELBO loss = Reconstruction + β * KL.
        """
        x_recon = forward_output['x_recon']
        gate_logits = forward_output['gate_logits']
        z_mean = forward_output['z_mean']
        z_logvar = forward_output['z_logvar']
        s = forward_output['s']  # This is the hard sample

        batch_size = x.shape[0]

        # 1. Reconstruction loss
        if self.reconstruction_loss_type == 'mse':
            recon_loss = F.mse_loss(x_recon, x, reduction='mean')
        elif self.reconstruction_loss_type == 'bce':
            recon_loss = F.binary_cross_entropy_with_logits(
                x_recon, x, reduction='mean'
            )
        else:
            raise ValueError(f"Unknown reconstruction loss: {self.reconstruction_loss_type}")

        # Posterior probability for gates, q(s=1|x)
        posterior_probs = torch.sigmoid(gate_logits)

        eps = 1e-8
        posterior_probs = posterior_probs.clamp(min=eps, max=1 - eps)

        # Fixed prior probability, p(s=1)
        prior_probs = float(self.prior_sparsity)
        prior_probs = max(min(prior_probs, 1 - eps), eps)

        # KL[q(s|x) || p(s)]
        kl_gates_per_dim = (
            posterior_probs * (torch.log(posterior_probs) - math.log(prior_probs))
            + (1 - posterior_probs) * (
                torch.log(1 - posterior_probs) - math.log(1 - prior_probs)
        	)
        )
        kl_gates = kl_gates_per_dim.sum(dim=1).mean()

        # GATED Slab KL: E_q(s|x)[KL[q(z-tilde|x) || p(zz-tilde)]]

        kl_slab_per_dim = -0.5 * (1 + z_logvar - z_mean.pow(2) - z_logvar.exp())

        # Then weight it by the posterior gate probability
        gated_kl_slab = (posterior_probs * kl_slab_per_dim).sum(dim=1).mean()

        # Total loss
        kl_total = kl_gates + gated_kl_slab

        total_loss = (
            recon_loss
            + beta * kl_total
        )

        return {
            'loss': total_loss,
            'recon_loss': recon_loss,
            'kl_gates': kl_gates,
            'kl_continuous_gated': gated_kl_slab,
            'kl_total': kl_total,
            'sparsity': (s.sum(dim=1).mean() / self.latent_dim),
            'avg_posterior_prob': posterior_probs.mean(),
        }

    def get_active_features(self, x: torch.Tensor, threshold: float = 0.5) -> torch.Tensor:
        """
        Get binary mask of active features for interpretation.
        """
        encoder_output = self.encoder(x)
        probs = torch.sigmoid(encoder_output['gate_logits'])
        return (probs > threshold).float()

    def reconstruct_from_features(
        self,
        x: torch.Tensor,
        feature_mask: torch.Tensor,
    ) -> torch.Tensor:
        """
        Reconstruct using only specified features (for interpretability analysis).

        Args:
            x: Input data
            feature_mask: Binary mask [batch, latent_dim] indicating which features to use
        """
        encoder_output = self.encoder(x)
        s, z_tilde, z = self.encoder.sample(
            encoder_output['gate_logits'],
            encoder_output['z_mean'],
            encoder_output['z_logvar'],
            hard=True,
        )

        # Apply feature mask
        z_masked = z * feature_mask

        return self.decoder(z_masked)


def train_spike_slab_vae(
    model: SpikeAndSlabVAE,
    train_loader: torch.utils.data.DataLoader,
    num_epochs: int = 100,
    learning_rate: float = 1e-3,
    beta_schedule: Optional[callable] = None,
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    if beta_schedule is None:
        beta_schedule = lambda epoch: 1.0

    for epoch in range(num_epochs):
        model.train()
        total_loss = total_recon = total_kl = total_sparsity = 0.0
        beta = beta_schedule(epoch)

        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            if len(data.shape) > 2:
                data = data.view(data.shape[0], -1)

            optimizer.zero_grad()
            forward_output = model(data)
            loss_dict = model.compute_loss(data, forward_output, beta=beta)
            loss_dict['loss'].backward()
            optimizer.step()

            model.decoder.apply_constraints()

            total_loss += loss_dict['loss'].item()
            total_recon += loss_dict['recon_loss'].item()
            total_kl += loss_dict['kl_total'].item()
            total_sparsity += loss_dict['sparsity'].item()

        n_batches = len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"  Loss: {total_loss/n_batches:.4f}")
        print(f"  Recon: {total_recon/n_batches:.4f}")
        print(f"  KL: {total_kl/n_batches:.4f}")
        print(f"  Sparsity: {total_sparsity/n_batches:.4f}")
        print(f"  Beta: {beta:.4f}")